# 01 | Bronze Layer — Vehicle Sales Data Ingestion

**Project:** US Vehicle Sales Analytics Pipeline  
**Notebook:** `01_bronze_ingestion`  
**Source:** GoodCarBadCar — US Passenger Car Sales 2026  
**Technology:** Databricks Free Edition | PySpark | Delta Lake | Unity Catalog  
**Load Type:** Full Historical Snapshot / File-Based Ingestion

---

## 1. Objective

The Bronze layer is responsible for ingesting raw vehicle sales data from the source CSV into a Delta Lake table while preserving the original data structure.

The main goals are:

- Preserve raw source data without business transformations.
- Maintain ingestion metadata for tracking and troubleshooting.
- Prevent duplicate ingestion of identical source files.
- Support historical snapshots and future reprocessing.
- Provide a reliable source for Silver-layer transformations.

---

## 2. Source Information

| Property | Description |
|---|---|
| Source System | GoodCarBadCar |
| Source File | US Vehicle Sales 2026.csv |
| Source Format | CSV |
| Data Domain | US Passenger Car Sales |
| Reporting Year | 2026 |
| Available Data | January–September 2026 |
| Source Records | 92 vehicle models |
| Ingestion Method | Unity Catalog Volume |
| Target Format | Delta Lake |
| Load Strategy | Initial Full Load + Future Snapshots |

**Source Structure:**

`Model, Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec, 2026 YTD`

October–December values are currently unavailable.

---

## 3. Bronze Ingestion Architecture

**Source CSV**  
↓  
**Unity Catalog Volume**  
↓  
**PySpark Ingestion Notebook**  
↓  
**File Checksum Validation**  
↓  
**Ingestion Metadata Enrichment**  
↓  
**Bronze Delta Table**

**Target Table:** `workspace.bronze.raw_vehicle_sales`

---

## 4. Ingestion Process

### Step 1 — Read Source File
Read the CSV from the Unity Catalog Volume using PySpark.

### Step 2 — Generate Batch Metadata
Generate a unique batch ID for tracking each ingestion execution.

### Step 3 — Calculate File Checksum
Calculate the SHA-256 hash of the source file to identify identical file content.

### Step 4 — Duplicate File Check
Check whether the source file checksum already exists in the Bronze table.

- If the checksum exists, skip ingestion.
- If the checksum is new, continue processing.

### Step 5 — Add Ingestion Metadata

| Column | Purpose |
|---|---|
| batch_id | Unique ingestion batch identifier |
| source_file | Original source file name |
| source_file_hash | SHA-256 checksum |
| ingestion_timestamp | Timestamp of ingestion |

### Step 6 — Write to Delta Lake
Append source records and ingestion metadata to the Bronze Delta table.

---

## 5. Load Strategy

**Initial Full Load**

Ingest the available 2026 historical sales snapshot.

**Future Snapshot Loads**

When a revised source file becomes available:

- Calculate the checksum.
- Skip identical snapshots.
- Append changed snapshots to Bronze.
- Preserve previous source versions.

The Silver layer will identify new and revised model-month sales records using deduplication and Delta MERGE.

---

## 6. Data Validation

After ingestion, verify:

- Source row count matches the Bronze batch row count.
- Required columns exist.
- Metadata fields are populated.
- Reprocessing an identical file does not create duplicate records.
- The Bronze table is readable.

Business-level data quality checks will be implemented in the Silver layer.

---

## 7. Design Decisions

**Why Delta Lake?**  
Provides ACID transactions, table history, and reliable downstream processing.

**Why append rather than overwrite?**  
Preserves historical source snapshots for auditing and reprocessing.

**Why preserve the original columns?**  
Allows raw source values to be retained before type conversion and business transformations.

**Why use a file checksum?**  
Detects identical file content and supports basic ingestion idempotency.

**Why store batch metadata?**  
Makes it easier to trace source records back to ingestion executions.

---

## 8. Expected Output

**Delta Table:** `workspace.bronze.raw_vehicle_sales`

**Expected Initial Record Count:** 92

**Expected Behavior:**

- First execution: Ingest 92 records.
- Second execution with identical file: Skip ingestion.
- New or revised file: Append a new source snapshot.

**Note:** The checksum check provides basic duplicate detection. Production-grade ingestion would also use a dedicated file-processing control table and concurrency-safe execution controls.

---

## 9. Downstream Processing

The Bronze Delta table serves as the input for:

**Next Notebook:** `02_silver_transformation`

Silver will perform:

- Monthly column unpivoting
- Data type conversions
- Null handling
- Data quality validation
- Deduplication
- Incremental Delta MERGE


In [0]:
import hashlib 
import uuid 
import pyspark.sql.functions as F
source_path = "/Volumes/vehicle_sales/bronze/us_vehical_sales/US Vehicle Sales 2026.csv"

table_name = f"vehicle_sales.bronze.raw_vehicle_sales"

#Generate unique Batch_id 
batch_id = str(uuid.uuid4())

# calculate file checksum for duplicate detection 
with open(source_path, "rb") as f:
  file_hash = hashlib.sha256(f.read()).hexdigest()


#check whether this exact file was alreday processed 
already_loaded = False

if spark.catalog.tableExists(table_name):
  already_loaded = spark.table(table_name).filter(F.col("source_file_hash") == file_hash).limit(1).count() > 0

if already_loaded:
  print(f"File {source_path} was already processed. Skipping...")
else:
  # read raw csv - preserve source columns as string 
  raw_df = spark.read.format("csv").option("header", "true").option("inferSchema", "false").option("mode","FAILFAST").load(source_path)
  
  # ADD ingestion metadata 
  raw_df = raw_df.withColumn("batch_id", F.lit(batch_id)) \
                 .withColumn("source_file", F.lit(source_path.split("/")[-1])
                             ) \
                 .withColumn("source_file_hash", F.lit(file_hash))\
                .withColumn("ingest_ts", F.current_timestamp())
    
  
  # write to bronze table 
  bronze_df = raw_df
  (
      bronze_df.write
      .format("delta")
      .mode("append")
      .saveAsTable(table_name)
  )
  
  print(f"SUCCESS: {bronze_df.count()} rows ingested")
  print(f"Batch ID: {batch_id}")


In [0]:
%sql
select source_file,
    batch_id,
    COUNT(*) AS total_rows,
    MIN(ingest_ts) AS ingestion_time from vehicle_sales.bronze.raw_vehicle_sales
    group by source_file, batch_id;

In [0]:
%sql 
DROP TABLE IF EXISTS vehicle_sales.bronze.raw_vehical_sales;